In [1]:
%%bash
pip install jupysql duckdb-engine

In [2]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

dataset_path = "/home/ds/datasets/"
dataset_filename = "competencia_02_crudo.csv.gz"

%sql duckdb:////home/ds/datasets/competencia_02.duckdb

In [3]:
%%sql
create or replace table competencia_02_crudo as
select
    *
from read_csv_auto("{{dataset_path + dataset_filename}}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


In [4]:
%%sql
select
    foto_mes
    , count(*) as cantidad
from competencia_02_crudo
group by foto_mes
order by foto_mes

,foto_mes,cantidad
0,201901,123916
1,201902,125020
2,201903,125622
3,201904,126116
4,201905,126763
5,201906,128370
6,201907,129853
7,201908,131775
8,201909,133419
9,201910,135633


In [5]:
%%sql
create or replace table competencia_02 as
with periodos as (
    select distinct foto_mes from competencia_02_crudo
), clientes as (
    select distinct numero_de_cliente from competencia_02_crudo
), todo as (
    select numero_de_cliente, foto_mes from clientes cross join periodos
), clase_ternaria as (
    select
        t.numero_de_cliente
        , t.foto_mes
        , c.* exclude (numero_de_cliente, foto_mes)
        , if(c.numero_de_cliente is null, 0, 1) as mes_0
        , lead(mes_0, 1) over w as mes_1
        , lead(mes_0, 2) over w as mes_2
        , case
            when mes_1 = 0                then 'BAJA+1'
            when mes_1 = 1 and mes_2 = 0  then 'BAJA+2'
            when mes_1 = 1 and mes_2 = 1  then 'CONTINUA'
            else null
          end as clase_ternaria
    from todo t
    left join competencia_02_crudo c using (numero_de_cliente, foto_mes)
    window w as (partition by t.numero_de_cliente order by t.foto_mes)
) select
  * exclude (mes_0, mes_1, mes_2)
from clase_ternaria
where mes_0 = 1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


In [6]:
%%sql
PIVOT competencia_02
on clase_ternaria
USING count(numero_de_cliente)
GROUP BY foto_mes
ORDER BY foto_mes

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,201901,647,697,122572
1,201902,700,696,123624
2,201903,700,774,124148
3,201904,776,575,124765
4,201905,576,658,125529
5,201906,665,589,127116
6,201907,595,656,128602
7,201908,659,543,130573
8,201909,548,615,132256
9,201910,621,632,134380


In [7]:
%%sql
COPY (
    select * from competencia_02 where foto_mes between 201901 and 201912
) TO '/home/ds/datasets/datos_2019.parquet' (FORMAT PARQUET)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


In [8]:
!cp ~/datasets/datos_2019.parquet ~/buckets/b1/datasets/
!ls -lh ~/datasets/datos_2019.parquet ~/buckets/b1/datasets/datos_2019.parquet

-rwxrwxrwx 1 ds ds 237M Nov 17 09:29 /home/ds/buckets/b1/datasets/datos_2019.parquet
-rw-r--r-- 1 ds ds 237M Nov 17 09:29 /home/ds/datasets/datos_2019.parquet
